In [1]:
import json
import pandas as pd
import numpy as np
import plotly.express as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
def parse_lines(lines):
    data = []
    for  line in lines:
        x = json.loads(line)
        data.append(x)

    data = pd.DataFrame.from_dict(data, orient="columns")
    return data

In [3]:
with open("./2mm.txt") as f:
    lines = f.readlines()

data = parse_lines(lines)
data = data.set_index(["i", "j", "k", "l"])
data["wctpf"] = data["wctpf"] / 1000;
data["wctpf_label"] = data["wctpf"].map({0: "Fully Mitigated", 25: "Mitigated", 1000000: "Unmitigated"})
data["tpf"] = data["dt"] / data["df"]
data["avg_tpf"] = data["acc_t"] / data["acc_f"]
data["frozen"] = data["irq"].apply(lambda x: True if "Freeze" in x else False)
data

timestamp   fuel      wctpf      dt     df      acc_t    acc_f  \
i j k  l                                                                     
0 0 0  0      6166081  10000  1000000.0       0      0          0        0   
       1    106489228  10000  1000000.0  259991  10000     259991    10000   
       2    106585250  10000  1000000.0   93226  10000     353217    20000   
       3    106679879  10000  1000000.0   94439  10000     447656    30000   
       4    106775680  10000  1000000.0   95641  10000     543297    40000   
...               ...    ...        ...     ...    ...        ...      ...   
  2 99 939  288532014  10000        0.0  100961  10000  188195476  9390000   
       940  288633566  10000        0.0  101442  10000  188296918  9400000   
       941  288734997  10000        0.0  101321  10000  188398239  9410000   
       942  288836259  10000        0.0  101151  10000  188499390  9420000   
       943  288940416  10000        0.0  104017   9885  188603407  9429885   

                        irq      wctpf_label        tpf    avg_tpf  frozen  
i j k  l                                                                    
0 0 0  0    {'Unfreeze': 1}      Unmitigated        NaN        NaN   False  
       1    {'Unfreeze': 1}      Unmitigated  25.999100  25.999100   False  
       2    {'Unfreeze': 1}      Unmitigated   9.322600  17.660850   False  
       3    {'Unfreeze': 1}      Unmitigated   9.443900  14.921867   False  
       4    {'Unfreeze': 1}      Unmitigated   9.564100  13.582425   False  
...                     ...              ...        ...        ...     ...  
  2 99 939    {'Freeze': 1}  Fully Mitigated  10.096100  20.042117    True  
       940    {'Freeze': 1}  Fully Mitigated  10.144200  20.031587    True  
       941    {'Freeze': 1}  Fully Mitigated  10.132100  20.021067    True  
       942    {'Freeze': 1}  Fully Mitigated  10.115100  20.010551    True  
       943    {'Freeze': 1}  Fully Mitigated  10.522711  20.000605    True  

[283200 rows x 12 columns]

In [4]:
tpf=data.reset_index()
tpf[["wctpf", "tpf"]].groupby("wctpf").mean()


# plt.box(tpf[tpf["k"] == 40][tpf["tpf"] < 11], x="tpf", color="wctpf", log_x=True)


plt.box(tpf[tpf["k"] == 40][tpf["tpf"] > 100][tpf["wctpf"] != 25], x="tpf", color="wctpf_label", labels={
    "tpf": "Instantanious Execution-Time per Fuel Unit [ns/fuel]",
    "wctpf_label": ""
}, width=800, height=400)



/tmp/ipykernel_46742/560557983.py:8: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  plt.box(tpf[tpf["k"] == 40][tpf["tpf"] > 100][tpf["wctpf"] != 25], x="tpf", color="wctpf_label", labels={
/tmp/ipykernel_46742/560557983.py:8: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  plt.box(tpf[tpf["k"] == 40][tpf["tpf"] > 100][tpf["wctpf"] != 25], x="tpf", color="wctpf_label", labels={


In [74]:
avg_tpf=data.groupby(["i", "j", "k"]).tail(1).reset_index()
fig = plt.histogram(avg_tpf, x="avg_tpf", color="wctpf_label", histnorm='percent', nbins=200,
    labels={
        "avg_tpf": "Total Execution-Time per Fuel Unit [ns/fuel]",
        "wctpf_label": ""
        },width=800, height=400
)
fig.add_vline(x=25, line_dash="dot", annotation_text="Mitigation Target:<br>25 ns/fuel", annotation_position="top right")
fig.write_image("mitigation-hist.pdf")
fig

In [75]:
x=data.reset_index()
fig = plt.line(x[x["k"] == 0], x="acc_f", y="avg_tpf", color="wctpf_label", labels={"acc_f": "Computational Progress [fuel]", "avg_tpf": "Avg. Execution-Time per Fuel Unit [ns/fuel]", "wctpf_label": ""}, width=800, height=400)
fig.add_hline(y=25, line_dash="dot", annotation_text="Mitigation Target:<br>25 ns/fuel", annotation_position="bottom right")
fig.write_image("mitigation-time.pdf")
fig

In [7]:
x = data.reset_index()[["i", "k", "fuel", "timestamp", "acc_t"]]
x = x.groupby(["i", "k"]).agg(["first", "last"])
x["eff"] = x["acc_t"]["last"] / (x["timestamp"]["last"] - x["timestamp"]["first"])
x=x.groupby("i").mean()
plt.line(x=x["fuel"]["first"], y=x["eff"], labels={"x": "Fuel-Chunk Size", "y": "Efficiency"})

In [77]:
x=data.reset_index()
fig=plt.line(x[(x["k"] == 0) & (x["wctpf"] == 25)], x="acc_f", y="frozen", color="wctpf_label", labels={"acc_f": "Computational Progress [fuel]", "frozen": "Is Mitigation Active", "wctpf_label": ""}, width=800, height=400).update_traces({"fill": "tozeroy", "line": {"color": "orangered"}})
fig.write_image("mitigation-active.pdf")
fig